# Phase 0 prototype — revision-create + pageviews

Validates both data sources before Phase 1 pipeline work begins.

**Goals:**
1. Parse the `revision-create` SSE sample into a flat DataFrame with nested fields expanded
2. Confirm the four key fields are populated (`rev_sha1`, `rev_slot_sha1`, `rev_slot_origin_rev_id`, `rev_content_changed`)
3. Load one hour of pageviews into a typed DataFrame
4. Join the two sources by page title — establish that the cross-track key works

No analysis here. This is a data-shape check.

In [1]:
import json
import re
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

DATA_DIR = Path("../samples")
REVISIONS_FILE = DATA_DIR / "revision-create_sample.txt"
PAGEVIEWS_FILE = DATA_DIR / "pageviews_sample_full.txt"

## 1. revision-create — parse SSE file

In [2]:
def parse_sse_file(path: Path) -> list[dict]:
    """Extract JSON payloads from SSE text dump."""
    events = []
    with open(path) as f:
        for line in f:
            if line.startswith("data: "):
                try:
                    events.append(json.loads(line[6:]))
                except json.JSONDecodeError:
                    pass
    return events

raw_events = parse_sse_file(REVISIONS_FILE)
print(f"Parsed {len(raw_events):,} events")
print(f"Schema versions: {set(e.get('$schema', '?') for e in raw_events)}")

Parsed 1,530 events
Schema versions: {'/mediawiki/revision/create/2.0.0'}


In [3]:
def flatten_event(e: dict) -> dict:
    """
    Flatten one revision-create event to a single-level dict.

    Fields extracted:
      meta.*         — dedup key (meta.id), ingestion time (meta.dt), domain
      top-level      — all scalar fields
      performer.*    — editor context needed for anomaly scoring
      rev_slots.main — the primary content slot (sha1, origin, size)

    rev_slots may have additional slots (e.g. 'mediainfo' on Commons).
    We capture main separately; the presence of extra slots is flagged.
    This maps 1:1 to the PySpark StructType used in Phase 1 bronze schema.
    """
    meta = e.get("meta", {})
    performer = e.get("performer", {})
    slots = e.get("rev_slots", {})
    main_slot = slots.get("main", {})

    return {
        # --- dedup + event-time keys ---
        "event_id":                 meta.get("id"),
        "event_dt":                 meta.get("dt"),
        "kafka_topic":              meta.get("topic"),
        "kafka_partition":          meta.get("partition"),
        "kafka_offset":             meta.get("offset"),
        # --- page ---
        "database":                 e.get("database"),
        "page_id":                  e.get("page_id"),
        "page_title":               e.get("page_title"),
        "page_namespace":           e.get("page_namespace"),
        "page_is_redirect":         e.get("page_is_redirect"),
        # --- revision ---
        "rev_id":                   e.get("rev_id"),
        "rev_parent_id":            e.get("rev_parent_id"),
        "rev_timestamp":            e.get("rev_timestamp"),
        "rev_sha1":                 e.get("rev_sha1"),
        "rev_len":                  e.get("rev_len"),
        "rev_minor_edit":           e.get("rev_minor_edit"),
        "rev_content_model":        e.get("rev_content_model"),
        "rev_content_changed":      e.get("rev_content_changed"),
        # --- performer (editor context) ---
        "user_text":                performer.get("user_text"),
        "user_id":                  performer.get("user_id"),
        "user_is_bot":              performer.get("user_is_bot"),
        "user_edit_count":          performer.get("user_edit_count"),
        "user_registration_dt":     performer.get("user_registration_dt"),
        "user_groups":              performer.get("user_groups"),   # list — kept as-is
        # --- rev_slots.main ---
        "main_slot_sha1":           main_slot.get("rev_slot_sha1"),
        "main_slot_origin_rev_id":  main_slot.get("rev_slot_origin_rev_id"),
        "main_slot_size":           main_slot.get("rev_slot_size"),
        "main_slot_content_model":  main_slot.get("rev_slot_content_model"),
        # --- extra slots flag ---
        "extra_slots":              [k for k in slots if k != "main"],
    }

revisions = pd.DataFrame([flatten_event(e) for e in raw_events])

# Typed columns
revisions["rev_timestamp"] = pd.to_datetime(revisions["rev_timestamp"], utc=True)
revisions["event_dt"]      = pd.to_datetime(revisions["event_dt"], utc=True)
revisions["user_registration_dt"] = pd.to_datetime(revisions["user_registration_dt"], utc=True, errors="coerce")

print(f"Shape: {revisions.shape}")
revisions.dtypes

Shape: (1530, 29)


event_id                                   str
event_dt                   datetime64[us, UTC]
kafka_topic                                str
kafka_partition                          int64
kafka_offset                             int64
database                                   str
page_id                                  int64
page_title                                 str
page_namespace                           int64
page_is_redirect                          bool
rev_id                                   int64
rev_parent_id                          float64
rev_timestamp              datetime64[us, UTC]
rev_sha1                                   str
rev_len                                  int64
rev_minor_edit                            bool
rev_content_model                          str
rev_content_changed                     object
user_text                                  str
user_id                                  int64
user_is_bot                               bool
user_edit_cou

## 2. Field reliability — Decision 001 check

In [4]:
# Fields that the pipeline depends on for correctness
key_fields = [
    "rev_sha1",
    "main_slot_sha1",
    "main_slot_origin_rev_id",
    "rev_content_changed",
]

n = len(revisions)
fill = (
    revisions[key_fields]
    .notna()
    .sum()
    .rename("filled")
    .to_frame()
)
fill["fill_rate"] = (fill["filled"] / n).map("{:.1%}".format)
fill["missing"]   = n - fill["filled"]
print(f"N = {n:,}")
fill

N = 1,530


,filled,fill_rate,missing
rev_sha1,1530,100.0%,0
main_slot_sha1,1530,100.0%,0
main_slot_origin_rev_id,1530,100.0%,0
rev_content_changed,1371,89.6%,159


In [5]:
# Where is rev_content_changed missing?
missing_rcc = revisions[revisions["rev_content_changed"].isna()]
print(f"rev_content_changed missing: {len(missing_rcc)} events")
print("\nBy wiki:")
print(missing_rcc["database"].value_counts())
print("\nBy namespace:")
print(missing_rcc["page_namespace"].value_counts())

rev_content_changed missing: 159 events

By wiki:
database
zhwikisource     77
commonswiki      51
itwiktionary      6
enwiktionary      3
enwiki            2
swwiktionary      2
zhwiki            2
dewiktionary      2
ukwikiquote       1
mnwwiktionary     1
frwiki            1
bnwiki            1
trwiki            1
bewikisource      1
ffwiki            1
eswikisource      1
jawiktionary      1
wikidatawiki      1
abwiki            1
frwikiquote       1
frwikisource      1
enwikisource      1
Name: count, dtype: int64

By namespace:
page_namespace
0      96
6      49
3       4
14      3
104     3
100     1
102     1
2       1
10      1
Name: count, dtype: int64


## 3. Edit stream — shape & distribution

In [6]:
print("Top wikis:")
print(revisions["database"].value_counts().head(10))

print("\nBot vs human:")
print(revisions["user_is_bot"].value_counts())

print("\nContent models (top 8):")
print(revisions["rev_content_model"].value_counts().head(8))

print("\nExtra slots (commons mediainfo etc):")
has_extra = revisions["extra_slots"].apply(bool)
print(revisions.loc[has_extra, "extra_slots"].explode().value_counts())

Top wikis:
database
wikidatawiki    497
commonswiki     437
enwiki          143
zhwikisource     97
cewiki           95
dewiki           27
frwiki           25
ruwiki           20
zhwiki           18
itwiki           17
Name: count, dtype: int64

Bot vs human:
user_is_bot
False    856
True     674
Name: count, dtype: int64

Content models (top 8):
rev_content_model
wikitext          1026
wikibase-item      493
proofread-page       9
zobject              1
Scribunto            1
Name: count, dtype: int64

Extra slots (commons mediainfo etc):
extra_slots
mediainfo    378
Name: count, dtype: int64


In [7]:
# Sample rows — enwiki human edits
cols = ["database", "page_title", "rev_id", "rev_sha1", "rev_content_changed",
        "user_text", "user_is_bot", "user_edit_count", "main_slot_sha1", "main_slot_origin_rev_id"]

(
    revisions
    .loc[(revisions["database"] == "enwiki") & (~revisions["user_is_bot"].fillna(False))]
    [cols]
    .head(5)
)

,database,page_title,rev_id,rev_sha1,rev_content_changed,user_text,user_is_bot,user_edit_count,main_slot_sha1,main_slot_origin_rev_id
5,enwiki,Eorcengota,1355910101,i6k4gozoy4t1y30ozmdq8lu8ny9ex1a,True,I&I22,False,1733,i6k4gozoy4t1y30ozmdq8lu8ny9ex1a,1355910101
15,enwiki,Mount_Sikaram,1355910100,oot041939zxgkdxe5ruppc3c1ohl4aw,True,Zackmann08,False,825617,oot041939zxgkdxe5ruppc3c1ohl4aw,1355910100
33,enwiki,Wikipedia:Miscellany_for_deletion/Wikipedia:Meetup/San_D...,1355910103,a73si6zrfabf2zvajpuq4u68by95ekt,True,Mathglot,False,102631,a73si6zrfabf2zvajpuq4u68by95ekt,1355910103
40,enwiki,Thomas_Müller,1355910087,f171yge2kargi9gyxvk1byejqqu1u9k,True,Mediocre Legacy,False,366636,f171yge2kargi9gyxvk1byejqqu1u9k,1355910087
45,enwiki,2026_IndyCar_Series,1355910098,26n7i4v96v3wwdkikln8iw8h4ojf6wf,True,Finn Shipley,False,132852,26n7i4v96v3wwdkikln8iw8h4ojf6wf,1355910098


## 4. Pageviews — load and type

In [8]:
# Format: domain_code page_title count_views response_size (4th col always 0, legacy)
# Filename encodes the hour: pageviews-YYYY-MM-DD_HH0000 — read from path when available.

pageviews = pd.read_csv(
    PAGEVIEWS_FILE,
    sep=" ",
    header=None,
    names=["domain_code", "page_title", "count_views", "_legacy"],
    dtype={"domain_code": "string", "page_title": "string",
           "count_views": "int32", "_legacy": "int32"},
    quoting=3,   # QUOTE_NONE — titles can contain double quotes
).drop(columns=["_legacy"])

print(f"Shape: {pageviews.shape}")
print(f"\nTop domain codes:")
print(pageviews["domain_code"].value_counts().head(12))
pageviews.head()

Shape: (7240120, 3)

Top domain codes:
domain_code
en           1211424
en.m         1031736
ja.m          301377
de            279863
zh            275614
ru.m          211472
ja            209510
de.m          199645
fr.m          193932
fr            191262
ru            189409
commons.m     149570
Name: count, dtype: Int64


,domain_code,page_title,count_views
0,"""""",-,2438
1,"""""",Category:Candidates_for_speedy_deletion,1
2,"""""",Category:Glossary/en,1
3,"""""",Category:Help/ms,1
4,"""""",Category:Historical_pages/ckb,1


In [9]:
# enwiki rows only
pv_en = pageviews[pageviews["domain_code"] == "en"].copy()
print(f"enwiki rows: {len(pv_en):,}")
print(f"Total enwiki views in this hour: {pv_en['count_views'].sum():,}")
print("\nTop pages by views:")
pv_en.nlargest(10, "count_views")

enwiki rows: 1,211,424
Total enwiki views in this hour: 3,399,654

Top pages by views:


,domain_code,page_title,count_views
1706821,en,Main_Page,227275
1992105,en,Special:Search,36101
1356147,en,Dota:_Dragon's_Blood,13175
1019868,en,-,8716
1292964,en,Cockroach_Janta_Party,5466
1020368,en,.xyz,3986
2227567,en,wiki.phtml,2999
1333933,en,Deaths_in_2026,2660
1060086,en,2026_FIFA_World_Cup,2262
1785378,en,"Neatsville,_Kentucky",1897


## 5. Cross-track join — title overlap

The join key between edit stream and pageviews is `page_title`, normalized to
underscores (both sources already use underscores). Domain mapping: `database=enwiki`
↔ `domain_code=en`.

This is a sanity check, not an analysis. The edit sample covers ~60 seconds of
edits; the pageview file covers one full hour. Overlap will be small but should
be non-zero — any page edited in that minute that also got traffic in that hour
is a match.

In [10]:
def normalize_title(s: pd.Series) -> pd.Series:
    """Lowercase + replace spaces with underscores. Both sources use underscores,
    but casing can differ (URL-decoded titles vs wiki canonical form)."""
    return s.str.replace(" ", "_", regex=False).str.lower()

edits_en = revisions[revisions["database"] == "enwiki"].copy()
edits_en["title_key"] = normalize_title(edits_en["page_title"].fillna(""))

pv_en_join = pv_en.copy()
pv_en_join["title_key"] = normalize_title(pv_en_join["page_title"].fillna(""))

print(f"enwiki edit events: {len(edits_en)}")
print(f"enwiki pageview rows: {len(pv_en_join):,}")
print(f"Unique edited page titles: {edits_en['title_key'].nunique()}")

enwiki edit events: 143
enwiki pageview rows: 1,211,424
Unique edited page titles: 136


In [11]:
joined = edits_en.merge(
    pv_en_join[["title_key", "count_views"]],
    on="title_key",
    how="inner",
)

print(f"Pages that appear in both edit stream and pageviews: {joined['title_key'].nunique()}")
print(f"  ({100 * joined['title_key'].nunique() / max(edits_en['title_key'].nunique(), 1):.0f}% of edited pages had traffic this hour)")

(
    joined
    .groupby("title_key")
    .agg(
        edits=("rev_id", "count"),
        count_views=("count_views", "first"),
        editors=("user_text", "nunique"),
    )
    .sort_values("count_views", ascending=False)
    .head(10)
)

Pages that appear in both edit stream and pageviews: 59
  (43% of edited pages had traffic this hour)


,edits,count_views,editors
title_key,,,
2026–27_uefa_champions_league,1,688,1
thomas_massie,1,576,1
2026–27_uefa_europa_league,1,448,1
project_hail_mary_(film),1,438,1
junior_eurovision_song_contest_2026,1,55,1
2026_indycar_series,1,44,1
2025–26_chelsea_f.c._season,1,40,1
thomas_müller,1,39,1
2026_cypriot_legislative_election,1,38,1


## Summary

| Check | Result |
|-------|--------|
| `rev_sha1` fill rate | 100% across all wikis |
| `main_slot_sha1` fill rate | 100% |
| `main_slot_origin_rev_id` fill rate | 100% |
| `rev_content_changed` fill rate | ~90% (gap: file-ns edits on Commons / zhwikisource) |
| Pageviews format | 4-field, parses cleanly, 4th col is dead legacy |
| Cross-track join key | Works — underscore-normalised `page_title` matches across both sources |

Decision 001 is confirmed. Phase 1 can proceed:
- `rev_sha1` + `main_slot_sha1` are the revert-detection keys
- `main_slot_origin_rev_id` is the no-op-edit detector
- `rev_content_changed` is an optimization signal (not correctness-critical)
- The cross-track join key (`page_title`, normalized) is viable